# 15.6 Token 怎麼送出再排回來？


原本輸入三個token，輸出也應保留三列、順序不變。但MoE會先按expert分組工作，工作完成的順序可能是「第2列、第0列、又一份第2列」。如果照完成順序直接排成輸出，就把詞的位置弄亂；如果同一列被兩位expert處理時後者覆蓋前者，又會丟失[加權合併](https://birdhackor.github.io/tiny-perceptron-vlm/15.5.html)的其中一份。送出時需要記住來源列號，收回時按列號相加。

我們希望收回後仍是原來三列，而且同列的兩份不能漏掉。用三筆已經乘好router比例的貢獻來看：來源列號是 `[2,0,2]`，貢獻向量分別為 `[1,1]`、`[2,2]`、`[3,3]`。先建立三列全零的結果。第一筆加到第2列，第二筆加到第0列，第三筆再次加到第2列；最後第0列 `[2,2]`，第1列 `[0,0]`，第2列 `[4,4]`。原列號從0開始，數字2表示第三列。

下圖左側按完成順序列出三份貢獻，箭頭依「列0或列2」送往右側原token位置。兩條箭頭匯入列2表示相加，不是覆蓋；列1沒有箭頭，這個刻意省略工作的例子中保持全零。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-15-row-restore.svg")))

程式用 `index_add_` 實作圖中的加回動作。`index_add_`沿指定軸依索引累加；第一個引數0指列方向，rows給每筆的目的列，contributions給要加的向量。名稱尾端 `_` 表示它直接修改result，沒有另外建立回傳結果。


In [ ]:
import torch

rows = torch.tensor([2, 0, 2])
contributions = torch.tensor([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]])
result = torch.zeros(3, 2)
result.index_add_(0, rows, contributions)
print("相加結果", result.tolist())
overwritten = torch.zeros(3, 2)
for row, value in zip(rows, contributions):
    overwritten[row] = value
print("覆蓋結果", overwritten.tolist())

第一行應是 `[[2,2],[0,0],[4,4]]`。後面的迴圈刻意用賦值，第三筆取代第一筆，所以第二行最後一列只有 `[3,3]`。這個反例讓我們看見，同樣完成三筆工作，合併方式錯了仍會丟數字。

前面的「送出」通常用索引選出某位expert負責的輸入列，稱為dispatch；這裡的「依原列號加回」稱為combine。rows記錄的是token身份，不是expert身份。即使同一expert收到很多token，它的每一份輸出仍要回到各自來源列；同一token被多位選中，才會有多筆回到同列。

本例第1列保持零，是因為我們沒有給它任何貢獻；不是說正常MoE應忽略第二個token。本教材使用dropless，意思是所有選中的工作都會執行，不因某位expert太忙而丟棄。有容量上限的設計則要另外處理溢出，後面[容量不足](https://birdhackor.github.io/tiny-perceptron-vlm/15.12.html)會區分這件事。現在先確保分組與收回沒有改變原有的計算內容。

練習把rows改成 `[1,0,1]`，先預測第1列變成 `[4,4]`、第2列全零，再重跑兩條路徑。接著交換第一筆與第三筆的排列，核對相加結果不變，但覆蓋結果變成 `[1,1]`。正確合併不應依完成先後而漏掉一份貢獻。

<details>
<summary>選讀：來源、量測條件與原始實報</summary>

正式訓練用的`MoEFFN`也先找出各expert的來源列，僅執行被選中的expert，再以`index_add_`合併。這是一個方便閱讀的Python分派迴圈，並未使用專用稀疏GPU核心。同批故事要補成相同長度才能組成整齊的輸入表，這個動作叫padding，補入的佔位符號叫PAD，可回看[有效輸入位置](https://birdhackor.github.io/tiny-perceptron-vlm/15.4.html)。目前expert的運算仍受到這張補齊後的表的形狀影響，沒有因為統計忽略PAD就自動跳過它們的工作。路由統計排除這些佔位；[輔助損失](https://birdhackor.github.io/tiny-perceptron-vlm/15.9.html)也排除它們，這是鼓勵分派較均衡的額外訓練項，避免把補齊工作算成真實輸入的負載。它是否值得使用，要同時看[15.11](https://birdhackor.github.io/tiny-perceptron-vlm/15.11.html)的實測成本與[15.13](https://birdhackor.github.io/tiny-perceptron-vlm/15.13.html)的品質，不能由資料流畫得正確就推論已加速。

</details>
